### Marketing data cleaning, sense-checking and initial EDA
**Goal**: Clean and sense-check a marketing dataset to be used for EDA, predictive modelling and segmentation.

**Context**:  This is a thorough and iterative investigation of the data, where insights from later sections may loop back to refine earlier steps. This notebook is intended for exploratory data cleaning and analysis, not for productionising a data pipeline with datasets that have predefined expectations. Much of the EDA is performed during validation and sense-checking, with additional charts at the end of the notebook.

In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from chart_utils import set_pomegranate_theme, POMEGRANATE_PALETTE, plot_100pct_stacked_hbar, plot_100pct_stacked_hbar_by_group, plot_distribution_bar, show_plot
set_pomegranate_theme()

In [0]:
base_dir = Path.cwd()  
output_dir = base_dir.parent / "output"
csv_path = base_dir.parent / "data" / "raw" / "ml_project1_data.csv"
marketing_raw = pd.read_csv(csv_path)

In [0]:
print(marketing_raw.shape, end='\n\n')
print(marketing_raw.dtypes)

In [0]:
marketing_raw.head(10)

In [0]:
marketing_raw.describe()

Check for duplicates, duplicate IDs and duplicate rows when stripped of IDs:

In [0]:
any_dupes = marketing_raw.duplicated().any()
duplicate_ID = marketing_raw['ID'].duplicated().any()
dupes_ex_ID = marketing_raw.drop(columns='ID').duplicated().any()

print(f'Has duplicates: {any_dupes}')
print(f'Duplicated IDs: {duplicate_ID}')
print(f'Any dupes when ID dropped: {dupes_ex_ID}')

There are some rows which are identical _except_ for their `ID`, which is likely to indicate an error somewhere in the process that produced this dataset.  How many rows?

In [0]:
num_dup_rows_ex_id = marketing_raw.drop(columns='ID').duplicated().sum()
print(num_dup_rows_ex_id)
print('{:.1%}'.format(num_dup_rows_ex_id / marketing_raw.shape[0]))

182, just over 8%.  We can see from the `marketing_raw.head(10)` above that many columns have highly precise entries (including `Income`), so it is extremely likely that these are true dupes, not different individuals with identical data. (In real life, I'd start a side quest to find out how these happened and improve the process.) Drop these near duplicates, keeping the lowest `ID`.

In [0]:
marketing_deduped = marketing_raw.drop_duplicates(subset=[c for c in marketing_raw.columns if c != 'ID']).reset_index(drop=True)

In [0]:
na_counts = marketing_deduped.isna().sum()
na_counts[na_counts > 0]

`Income` is oddly precise; I would expect a marketing survey to have ranges for income, and even if they didn't I would expect more rounded estimates or NAs.  There are 24 NAs and an implausible 666,666 max value.  
1. Replace the NAs with a dummy value for now, to facilitate grouping in the course of cleaning, and I don't want to replace/impute NAs yet
2. Also replace the 666,666 value with the dummy - this is an extreme outlier and probably an 'I'm not telling you that' input from the customer 

In [0]:
dummy = -999
marketing_deduped['Income'] = marketing_deduped['Income'].fillna(dummy)
marketing_deduped.loc[marketing_deduped['Income'] == 666666, 'Income'] = dummy
marketing_deduped['Income'].describe()

Check the object fields next, first for empty or all-whitespace strings:

In [0]:
non_num_cols = marketing_deduped.select_dtypes(exclude=[np.number]).columns

counts = (
    marketing_deduped[non_num_cols]
      .apply(lambda s: s.astype('string').str.strip().eq('').sum())
)

print(counts)

Convert `Dt_Customer` (customer join date) into a proper datetime.  Rename it for clarity.

In [0]:
marketing_deduped['Dt_Customer'] = pd.to_datetime(marketing_deduped['Dt_Customer'], format='%Y-%m-%d', errors='coerce')
marketing_deduped.rename(columns={'Dt_Customer': 'Join_date'}, inplace=True)
print(marketing_deduped['Join_date'].dtype)
print(marketing_deduped['Join_date'].isna().sum())
print(marketing_deduped['Join_date'].describe())


It is not clear from the dataset and data dictionary just *when* the `Response` was gathered and when the 'original' analysis would have happened. As we can see above, the most recent customer sign-up date is mid-2014. Let's look at the earliest possible date that this data could have been gathered - the maximum of `Join_date` plus `Recency`. 

In [0]:
join_date_plus_recency = marketing_deduped['Join_date'] + pd.to_timedelta(marketing_deduped['Recency'], unit='D')
join_date_plus_recency.max()

The max(`Join_date`+`Recency`) is 4 October 2014, so we'll assume that the current year is 2014 and the current date 5 October 2014. I can use this for better human interpretability (eg to derive approximate `Age` from `Year_Birth`), and since the `Response` is not date-based, risk of data leakage is very low.

In [0]:
current_year = 2014
current_date = pd.to_datetime('2014-10-05')

Most of the fields would have been generated by customer purchases and events on the platform, but there are several fields which are presumably self-reported, so we will sense-check carefully:
* `Marital_Status`
* `Education`
* `Year_Birth`
* `Kidhome`
* `Teenhome`
* `Income`

In [0]:
marketing_deduped['Marital_Status'].value_counts(dropna=False)

Let's see these YOLO and Absurd rows.  

In [0]:
marketing_deduped[marketing_deduped['Marital_Status'].isin(['Absurd', 'YOLO'])]

Okay, I can accept that 0.1% of respondents independently wrote 'Absurd', but the YOLO rows look identical except for `ID` *and* `Response`! Are there any others?

In [0]:
num_dup_rows_ex_id_response = marketing_deduped.drop(columns=(['ID', 'Response'])).duplicated().sum()
print(num_dup_rows_ex_id_response)
print('{:.2%}'.format(num_dup_rows_ex_id_response / marketing_raw.shape[0]))

19 distinct rows would remain, less than 1% of the original dataset.

Now, since `Response` is the target to predict, the existence of contradictory responses from what is supposedly a single offer to each customer, is a major issue. We don't know how this happened or which `Response` is the true one, or even if the offer was somehow made twice to these respondents with one refusal and one acceptance.  (In real life, my side quest to investigate how this data was collected and recorded would move up my priority list at this point.) 

Dropping all these rows as unreliable would be 19*2 = 38,  or 1.7% of the original dataset, which is small, so let's just drop these.  

In [0]:
ignore = ['ID', 'Response']
col_subset = [c for c in marketing_deduped.columns if c not in ignore]

marketing_deduped = marketing_deduped.loc[
    ~marketing_deduped.duplicated(subset=col_subset, keep=False)
    ].reset_index(drop=True)

marketing_deduped.shape

Recheck `Marital_status`.

In [0]:
marketing_deduped['Marital_Status'].value_counts(dropna=False)

Fold 'Alone' and 'Absurd' into 'Single'. 

In [0]:
marketing_deduped['Marital_Status'] = marketing_deduped['Marital_Status'].replace(
    {'Absurd': 'Single', 'Alone': 'Single'})

Now `Education`:

In [0]:
marketing_deduped['Education'].value_counts(dropna=False)

Seems sensible.  These are standard education levels in Brazil, the original source of this data.  Convert to approximate years of education.


In [0]:
marketing_deduped['Education_years'] = np.select(
    [
        marketing_deduped['Education'].eq('Basic'),
        marketing_deduped['Education'].eq('Graduation'),
        marketing_deduped['Education'].eq('2n Cycle'),
        marketing_deduped['Education'].eq('Master'),
        marketing_deduped['Education'].eq('PhD'),
    ],
    [12, 16, 18, 18, 22],
    default=np.nan
).astype(int)

Now `Year_Birth`, with the assumption that the 'current year' is 2014, can be used to approximate `Age`. We saw in the `describe()` of the raw dataset that the minimum `Year_Birth` is 1893, or age 121, which is unlikely. The maximum `Year_Birth` is 1996, which for the probable 'current year' 2014 indicates age 18, which is perfectly sensible.  Let's see a quick distribution.

In [0]:
sns.boxplot(data=marketing_deduped, x='Year_Birth')
plt.show()

In [0]:
marketing_deduped[marketing_deduped['Year_Birth'] < 1940]

And two of these 100+ year olds have a kid or teen at home.  Replace these with the median.

In [0]:
median_all = int(np.rint(marketing_deduped['Year_Birth'].median()))

outliers = marketing_deduped['Year_Birth'] <= 1900
marketing_deduped.loc[outliers, 'Year_Birth'] = median_all
marketing_deduped['Year_Birth'].describe()
marketing_deduped['has_imputed_age'] = outliers.astype(int)

This looks very reasonable.  Drop the `Year_Birth`, as `Age` is more immediately understandable for our current purposes.  Then make some `Age` categories. 

In [0]:
marketing_deduped['Age'] = current_year - marketing_deduped['Year_Birth']
marketing_deduped['Age'].describe()

In [0]:
marketing_deduped = marketing_deduped.drop(columns=['Year_Birth'])

bins = [18,25,35,45,55,65,75]  
labels = ['18-24','25-34','35-44','45-54','55-64','65-74']

marketing_deduped['Age_category'] = pd.cut(marketing_deduped['Age'], bins=bins, right=False, labels=labels)

counts = marketing_deduped['Age_category'].value_counts(sort=False)

plt.figure(figsize=(7,4))
sns.barplot(x=counts.index.astype(str), y=counts.values)
plt.xlabel('Age categories')
plt.ylabel('Count of customers')
plt.savefig(output_dir / "age_categories_column.png", dpi=150, bbox_inches='tight')
plt.show()

Seems reasonable.

We can see above that `Kidhome` and `Teenhome` are both are integer types with min of 0 and maximum of 2. Quick visualisation: 

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(10,4), sharey=True)
sns.countplot(data=marketing_deduped, x='Kidhome', order=[0,1,2], ax=axes[0])
axes[0].set_title('Kids home')
sns.countplot(data=marketing_deduped, x='Teenhome', order=[0,1,2], ax=axes[1])
axes[1].set_title('Teens home')
plt.tight_layout()
plt.savefig(output_dir / "kidhome_teenhome_column.png", dpi=150, bbox_inches='tight')
plt.show()

Add three new fields: `Minorhome` (kids + teens), `AdultHome` (based on single/widowed/divorced vs married/together) and `HouseholdSize`.  The latter two are estimates, as other adults in the household aren't asked about.

In [0]:
marketing_deduped['Minorhome'] = marketing_deduped['Kidhome']+marketing_deduped['Teenhome']

marketing_deduped['Adulthome'] = np.where(
    marketing_deduped['Marital_Status'].isin(['Married', 'Together']), 2, 1).astype(int)

marketing_deduped['HouseholdSize'] = marketing_deduped['Adulthome']+marketing_deduped['Minorhome']

max_household = marketing_deduped['HouseholdSize'].max()
max_minors = marketing_deduped['Minorhome'].max()

fig, axes = plt.subplots(1, 3, figsize=(10,4), sharey=True)
sns.countplot(data=marketing_deduped, x='Minorhome', order=list(range(0, max_minors + 1)), ax=axes[0])
axes[0].set_title('Minors home')
sns.countplot(data=marketing_deduped, x='Adulthome', order=[1,2], ax=axes[1])
axes[1].set_title('Adults home')
sns.countplot(data=marketing_deduped, x='HouseholdSize', order=list(range(1, max_household + 1)), ax=axes[2])
axes[2].set_title('Household size')
plt.tight_layout()
plt.savefig(output_dir / "adults_minors_household_column.png", dpi=150, bbox_inches='tight')
plt.show()

Again, seems quite reasonable. Moving on to `Income`.

There are 24 NAs and one odd value which I replaced above with a dummy value.  Let's put the NAs back (marking them as such in case I wish to re-impute or drop later), impute them to the medians while grouping by marital status, age and education, then check the distribution.  

In [0]:
marketing_deduped['Income'] = marketing_deduped['Income'].replace(dummy, np.nan)
marketing_deduped['has_imputed_income'] = marketing_deduped['Income'].isna().astype(int)

grp = ['Marital_Status', 'Age_category', 'Education']
med = marketing_deduped.groupby(grp, observed=True)['Income'].transform('median')
marketing_deduped['Income'] = marketing_deduped['Income'].fillna(med)

print(f"Income NA count: {marketing_deduped['Income'].isna().sum()}")
print(marketing_deduped[['Income', 'has_imputed_income']].describe())
print(marketing_deduped[['Income', 'has_imputed_income']].dtypes)

The original dataset is from Brazil, and we have reason to believe the that purported data collection was from 2014.  It seems reasonable that the numbers are in Brazil's currency, the Brazilian real (BRL), although this is not stated explicitly. 

Just for sense-checking, I'd like to compare these figures with other data on Brazilian household income.  This is just a quick ballpark check and I'm only using free sources, so I'm settling for data gathered 2006-2012.  

Gallup has figures for median annual household income in Brazil, data gathered 2006-2012, and stated in US dollars: **USD 7,552**  
source: https://news.gallup.com/poll/166211/worldwide-median-household-income-000.aspx

Average exchange rate for calendar 2012: **BRL 0.5139 = USD 1** 
source: https://www.exchange-rates.org/exchange-rate-history/brl-usd-2012

So median household income in Brazil in 2012 was around **BRL 14,695**.

`Income` in this dataset has a median value of **BRL 52,013**.  This is much higher than the overall median from the Gallup survey, but this is to be expected.  Brazil's GINI coefficient (a measure of income inequality) is quite high, and the customers of this food delivery app (especially in 'current year' 2014) are more likely to be urbanites with higher incomes than the median Brazilian.  

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
sns.boxplot(data=marketing_deduped, x='Income', ax=axes[0])
axes[0].set_title('Income boxplot')
axes[0].tick_params(axis='both', labelsize=8)
sns.histplot(data=marketing_deduped, x='Income', kde=True, ax=axes[1], bins='auto')
axes[1].set_title('Income distribution (hist + KDE)')
axes[1].tick_params(axis='both', labelsize=8)
plt.tight_layout()
plt.savefig(output_dir / "income_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

There's an interesting set of outliers around the 150-160K level.  Quick look at these: 

In [0]:
marketing_deduped[marketing_deduped['Income'] > 120000]

I don't see any obvious reasons to be suspicious of these six rows, but the outlier thing is interesting.  I'll keep this in mind and may exclude or cap them later.

Now let's take a quick look at the rest of the fields, starting wiht booleans (0/1).  

In [0]:
boolean_columns = ['Complain', 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response'] 

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
axes = axes.flatten()
for i, col in enumerate(boolean_columns):
    ax = axes[i]
    sns.countplot(data=marketing_deduped, x=col, order=[0, 1], ax=ax) 
    ax.set_title(col, fontsize=10)    
    ax.set_xlabel('') 
    ax.tick_params(axis='x', labelsize=8)

# blank the last subplot
for j in range(len(boolean_columns), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

The five previous offers.... were they available to *all* customers in this set?  If not, may want to estimate a date of each offer and then create a feature (based on `Join_date`) which is the percentages possible offers, not a sum. Check the max join date of customers known to have accepted each of the five offers: 

In [0]:
accepted_cols = [c for c in marketing_deduped.columns if c.startswith("Accepted")]

accepted_long = pd.melt(marketing_deduped, id_vars=['ID','Join_date'], value_vars=accepted_cols, var_name='Campaign', value_name='Accepted')
last_joindate_each_campaign = accepted_long[accepted_long.Accepted == 1].groupby('Campaign')['Join_date'].max().reset_index()
last_joindate_each_campaign

Those are all reasonably close (2 weeks or less) of the latest join date of 29 June 2014, so assume that all offers were potentially offered to all customers in this set, therefore no need to take a percentage rather than sum.  

Now take a look at histograms of the rest of the columns:

In [0]:
integer_columns = ['Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts','MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases','NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']

fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey=True)
axes = axes.flatten()
for i, col in enumerate(integer_columns):
    ax = axes[i]    
    sns.histplot(data=marketing_deduped, x=col, bins='auto', ax=ax)    				
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')        
    ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
plt.show()

Add a few useful new fields:
* Sum of offers accepted (not counting response)
* Total amount spent
* Total number of purchases made
* Gold products spend as a percent o total spend - these are premium products
* Deal purchases as a percent total purchases - these are deals offered by the retailer
* Average spend per purchase
* Average spend per month that the customer has been a member - the spend amounts are for the last 2 years, so cap the 'time customer has been a member' at 2 years
* Average purchases per month, ditto

In [0]:
spend_cols = [c for c in marketing_deduped.columns if c.startswith("Mnt") & (c.endswith("GoldProds") == False)]
purchase_cols = [c for c in marketing_deduped.columns if c.endswith("Purchases") & (c.endswith("DealsPurchases") == False)]

marketing_deduped = (
    marketing_deduped
    .assign(
         TotalSpend=marketing_deduped[spend_cols].sum(axis=1),
         NumAccepted=marketing_deduped[accepted_cols].sum(axis=1),
         NumPurchases=marketing_deduped[purchase_cols].sum(axis=1),
         MonthsSinceJoiningCapped=((current_date - marketing_deduped["Join_date"]).dt.days).clip(upper=730) / (365/12),
         DaysSinceJoin = (current_date - marketing_deduped["Join_date"]).dt.days
     )
)
marketing_deduped[['TotalSpend', 'NumAccepted', 'NumPurchases', 'MonthsSinceJoiningCapped']].describe()

Hm.   The minimums of total purchases and total spend are both 0 - possible it might have happened due to a refund or similar.  Are there many?


In [0]:
marketing_deduped.loc[(marketing_deduped['TotalSpend'] == 0) | (marketing_deduped['NumPurchases'] == 0)]

Only six rows, all with tiny total spend, haven't accepted any offers and Response all 0.  Interestingly, two of these are part of the ~160K income outlier set.

Drop these, add more fields including percentages of Gold products spend and percentage of purchases with Deals.  then another quick visualisation and sense check.

In [0]:
marketing_deduped = (
    marketing_deduped
    .drop(marketing_deduped.index[(marketing_deduped['NumPurchases'] == 0) | (marketing_deduped['TotalSpend'] == 0)])
)

marketing_deduped = (
    marketing_deduped
    .assign(
         PercentGold=marketing_deduped['MntGoldProds'] / marketing_deduped['TotalSpend'],
         PercentDealPurchases=marketing_deduped['NumDealsPurchases'] / marketing_deduped['NumPurchases'],
         SpendPerPurchase=marketing_deduped['TotalSpend'] / marketing_deduped['NumPurchases'],
         SpendPerMonth=marketing_deduped['TotalSpend'] / marketing_deduped['MonthsSinceJoiningCapped'],
         PurchasesPerMonth=marketing_deduped['NumPurchases'] / marketing_deduped['MonthsSinceJoiningCapped'],
         SpendPercIncomeMonth=(marketing_deduped['TotalSpend'] / marketing_deduped['MonthsSinceJoiningCapped']) / (marketing_deduped['Income'] / 12) 
     )
)

marketing_deduped[['PercentGold', 'PercentDealPurchases', 'SpendPerPurchase', 'SpendPerMonth', 'PurchasesPerMonth', 'SpendPercIncomeMonth']].describe()

**Issues:**
- We shouldn't see percentages of more than 1  
- Spend per purchase of 1657 seems exceptionally high
- Likewise a total spend of 40% of monthly income implies a possible mistake

In [0]:
marketing_deduped.loc[(marketing_deduped['PercentGold'] > 1) | (marketing_deduped['PercentDealPurchases'] > 1)]

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=False)
sns.boxplot(data=marketing_deduped, x='SpendPerPurchase', ax=axes[0])
axes[0].set_title('Spend per purchase')
sns.boxplot(data=marketing_deduped, x='SpendPercIncomeMonth', ax=axes[1])
axes[1].set_title('Mean monthly spend as percentage of monthly income')
plt.tight_layout()
plt.show()

In [0]:
marketing_deduped.loc[(marketing_deduped['SpendPerPurchase'] > 250) | (marketing_deduped['SpendPercIncomeMonth'] > 0.1)]

- For the `SpendPercIncomeMonth` outlier, this is a long-term customer with two adults and a child - strongly suspect error in `Income` input, impute as previously
- For the `SpendPerPurchase` outlier: A single customer, one purchase, almost entirely meat, which is theoretically possible (one-off for large event?), but it's such an extreme outlier and not reflective of normal household behaviour, so will drop,  along with the percentages fields that are greater than 1

Also, since we've dropped several rows since we imputed `Income`, re-impute and recalculate `SpendPercIncomeMonth`.

In [0]:
mask = (
    (marketing_deduped["PercentDealPurchases"] > 1) |
    (marketing_deduped["PercentGold"] > 1) |
    (marketing_deduped["SpendPerPurchase"] > 250)
)
marketing_deduped = marketing_deduped.drop(marketing_deduped.index[mask])

marketing_deduped.loc[marketing_deduped['SpendPercIncomeMonth'] > 0.4, ['Income', 'has_imputed_income', 'SpendPercIncomeMonth']] = [np.nan, 1, np.nan]

marketing_deduped.loc[marketing_deduped['has_imputed_income'] == 1, ['Income']] = [np.nan]

grp = ['Marital_Status', 'Age_category', 'Education']
med = marketing_deduped.groupby(grp, observed=True)['Income'].transform('median')
marketing_deduped['Income'] = marketing_deduped['Income'].fillna(med)

marketing_deduped['SpendPercIncomeMonth']=(marketing_deduped['TotalSpend'] / marketing_deduped['MonthsSinceJoiningCapped']) / (marketing_deduped['Income'] / 12) 

print(f"Income NA count: {marketing_deduped['Income'].isna().sum()}")
print(marketing_deduped[['Income', 'SpendPercIncomeMonth']].describe())


In [0]:

new_cols = ['TotalSpend', 'NumAccepted', 'NumPurchases', 'PercentGold', 'PercentDealPurchases', 'SpendPerPurchase', 'SpendPerMonth', 'PurchasesPerMonth', 'SpendPercIncomeMonth']

fig, axes = plt.subplots(3, 3, figsize=(14, 6), sharey=False)
axes = axes.flatten()
for i, col in enumerate(new_cols):
    ax = axes[i]
    sns.histplot(data=marketing_deduped, x=col, ax=ax) 
    ax.set_title(col, fontsize=10)    
    ax.set_xlabel('') 
    ax.tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.show()

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3.3))

sns.histplot(data=marketing_deduped, x='Join_date', bins=8, ax=ax)

ax.set_title('Customer join date', fontsize=10)
ax.tick_params(axis='x', labelsize=8)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Summary of the event data:
* The boolean columns, which in whether customer has complained within last 2 years, and acceptance of five previous  offers plus the Response of the most recent offer): small number of 'yes' for all
* The amounts spent on various sorts of products purchased has a right-skewed distribution, typical of retail purchase data. Meats and Wines are by far the biggest spends
* The number of purchases by various channels; since not everyone is buying through ALL channels, there are a small number of 0s, otherwise right-skewed, also typical
* Number of web visits in the last month; a very few with between 10 and 20, mode is 8 - twice a week seems a pretty normal shopping pattern
* Both recency (number of days since last purchase) and customer join date are fairly evenly distributed

So this data is now reasonably clean and makes sense.

Next steps (such as insights analysis, segmenting, prediction model(s)) may require further processing, and may take place in other environments, so save this as a CSV.


In [0]:
out_path = base_dir.parent / "data" / "processed" / "marketing_clean.csv"
marketing_deduped.to_csv(out_path, index=False)

In [0]:
numeric_cols_subset1 = ['Income', 'TotalSpend', 'NumWebVisitsMonth','NumPurchases', 'NumWebPurchases', 'NumCatalogPurchases',
       'NumStorePurchases', 'MntWines', 'MntFruits', 'MntMeatProducts',
       'MntFishProducts', 'MntSweetProducts', 'PercentGold', 'PercentDealPurchases', 'SpendPerPurchase', 'SpendPerMonth', 'PurchasesPerMonth', 'SpendPercIncomeMonth', 'Recency', 'DaysSinceJoin','Education_years', 'Age', 'HouseholdSize', 'Adulthome', 'Minorhome', 'Kidhome', 'Teenhome', 'Complain', 'NumAccepted', 'Response']

num_subset_df = marketing_deduped[numeric_cols_subset1]

correlation_matrix = num_subset_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, 
            annot=True, 
            cmap='coolwarm', # for this one just easier to read than tweaking my custom palette
            fmt=".2f", 
            annot_kws={'size': 6},
            linewidths=0.5)
plt.title('Correlation matrix')
plt.savefig(output_dir / "correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


* The single calculated field most highly correlated with `Response` is the number of previous offers accepted
* A positive `Response` is also associated with higher `TotalSpend`, number of purchases overall as well as spend per purchase, a smaller household size, long-standing customers (higher `DaysSinceJoin`) and more recent purchases (lower `Recency`)

In [0]:
numeric_cols_subset2 = ['Income', 'TotalSpend', 'NumPurchases', 'PercentGold', 'PercentDealPurchases', 'SpendPerPurchase', 'Recency', 'DaysSinceJoin', 'SpendPerMonth', 'PurchasesPerMonth', 'Response']

num_subset2_df = marketing_deduped[numeric_cols_subset2 ]
numeric_cols_subset2_ex_response = [col for col in numeric_cols_subset2 if col != 'Response']

fig, axes = plt.subplots(2, 5, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols_subset2_ex_response):
    sns.kdeplot(data=num_subset2_df, x=col, hue='Response', ax=axes[idx], linewidth=2, fill=True)
    
    axes[idx].set_title(col, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(output_dir / "selected_distributions_by_reponse.png", dpi=150, bbox_inches='tight')
plt.show()

In [0]:
sns.pairplot(num_subset_df[numeric_cols_subset2], 
             diag_kind='kde',           # Kernel density on diagonal
             plot_kws={'alpha': 0.2},   # Transparency for overlapping points
             hue='Response')
plt.tight_layout()
plt.show()


In [0]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes_flat = axes.ravel()

colchart_cols = [
    'Complain', 'Education_years', 'Minorhome', 'Adulthome',
    'Age_category', 'HouseholdSize', 'NumWebVisitsMonth',
    'NumWebPurchases', 'NumAccepted'
]

colchart_df = marketing_deduped[colchart_cols].assign(
    Complain=marketing_deduped['Complain'].astype(str),
    Education_years=marketing_deduped['Education_years'].astype(str),
    Minorhome=marketing_deduped['Minorhome'].astype(str),
    Adulthome=marketing_deduped['Adulthome'].astype(str),
    NumAccepted=marketing_deduped['NumAccepted'].astype(str),
    HouseholdSize=marketing_deduped['HouseholdSize'].astype(str),
    NumWebVisitsMonth=marketing_deduped['NumWebVisitsMonth'].astype(str).str.zfill(2),
    NumWebPurchases=marketing_deduped['NumWebPurchases'].astype(str).str.zfill(2)
)

for idx, col in enumerate(colchart_cols):
    ct = pd.crosstab(
        colchart_df[col],
        marketing_deduped['Response'].astype(bool),
        normalize='index'
    ) * 100

    ct.plot(kind='bar', stacked=True, ax=axes_flat[idx])

    axes_flat[idx].set_title(col, fontweight='bold')
    axes_flat[idx].set_ylabel('Percentage')
    axes_flat[idx].set_xlabel('')
    axes_flat[idx].legend(title='Response', fontsize=8, loc='lower left')
    axes_flat[idx].grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig(output_dir / "selected_100pc_by_reponse.png", dpi=150, bbox_inches='tight')
plt.show()


In [0]:
# counts and %
counts = (
    marketing_deduped
    .groupby(["Age_category", "Adulthome"])["Response"]
    .value_counts(dropna=False)
    .unstack(fill_value=0)
    .rename(columns={0: "Response_0", 1: "Response_1"})
)

# add totals + percentages
counts["Total"] = counts.sum(axis=1)
counts["Response_0_pct"] = (counts["Response_0"] / counts["Total"] * 100).round(2)
counts["Response_1_pct"] = (counts["Response_1"] / counts["Total"] * 100).round(2)

# order columns nicely
counts = counts[["Response_0", "Response_1", "Total", "Response_0_pct", "Response_1_pct"]]

counts.reset_index(inplace=True)
counts


According to the  data dictionary, NumWebVisitsMonth are only for the last  month, where NumWebPurchases (and all the other purchase counts!) are over an unspecified time period, so these don't sense-check well against each other (web visits in last month could be 0 but could have a lot of prior web purchases).  However, let's look at how they relate to each other and to Response (excluding a very few outliers).

In [0]:
agg = (marketing_deduped
       .groupby(["NumWebVisitsMonth", "NumWebPurchases", "Response"])
       .size()
       .reset_index(name="n"))

palette = {0: '#c41e3a', 1: '#004e7c'} 

plt.figure(figsize=(10, 8))

scatter = sns.scatterplot(
    data=agg,
    x="NumWebVisitsMonth",
    y="NumWebPurchases",
    hue="Response",
    size="n",
    palette=palette,
    hue_order=[0, 1],
    sizes=(50, 1000),  
    alpha=0.7,
    edgecolor="w",
    linewidth=0.5
)

plt.legend(title="Response", loc="upper right")
handles, labels = scatter.get_legend_handles_labels()

plt.xlim(0, 12)
plt.ylim(0, 12)
plt.xlabel("Web visits")
plt.ylabel("Web purchases")
plt.title("Web Visits vs Web Purchases by Response (Bubble Size = Count)")

plt.tight_layout()
plt.show()

Facet to make some of the differences clearer. NOte that i this chart, the bubble sizes are scaled per facet, not overall.

In [0]:
g = sns.FacetGrid(
    agg, col="Response", col_wrap=2,
    height=4, aspect=1,
    sharex=True, sharey=True
)

g.map_dataframe(
    sns.scatterplot,
    x="NumWebVisitsMonth",
    y="NumWebPurchases",
    hue="Response",
    palette=palette,
    hue_order=[0, 1],
    size="n",
    sizes=(20, 800),     
    alpha=0.7,
    legend=False
)

g.set(xlim=(0, 12), ylim=(0, 12))
g.set_axis_labels("Web visits", "Web purchases")

plt.tight_layout()
plt.savefig(output_dir / "web_bubbles_by_reponse.png", dpi=150, bbox_inches='tight')
plt.show()



In [0]:
marketing_deduped[marketing_deduped['NumWebPurchases'] > 15]

### Horizontal 100% bar charts 
...instead of pie charts :)

In [0]:
fig, ax = plot_100pct_stacked_hbar(marketing_deduped, spend_cols)
ax.set_title("Total spend by product type")
plt.savefig(output_dir / "spend_by_product.png", dpi=150, bbox_inches='tight')
show_plot(fig, ax)

In [0]:
fig, ax = plot_100pct_stacked_hbar_by_group(marketing_deduped, spend_cols, group_col="Response")
ax.set_title("Total spend split by product type and response")
plt.savefig(output_dir / "spend_by_product_and_reponse.png", dpi=150, bbox_inches='tight')
show_plot(fig, ax)

In [0]:
fig, ax = plot_100pct_stacked_hbar(marketing_deduped, purchase_cols)
ax.set_title("Total spend by channel")
plt.savefig(output_dir / "spend_by_channel.png", dpi=150, bbox_inches='tight')
show_plot(fig,ax)

In [0]:
fig, ax = plot_100pct_stacked_hbar_by_group(marketing_deduped, purchase_cols, group_col='Response')
ax.set_title("Total spend by channel and response")
plt.savefig(output_dir / "spend_by_channel_and_response.png", dpi=150, bbox_inches='tight')
show_plot(fig,ax)

In [0]:
fig, ax = plot_100pct_stacked_hbar_by_group(marketing_deduped, spend_cols, group_col='Age_category')
ax.set_title("Total spend by product and age category")
plt.savefig(output_dir / "spend_by_product_age_category.png", dpi=150, bbox_inches='tight')
show_plot(fig,ax)

In [0]:
fig, ax = plot_distribution_bar(marketing_deduped, "Response")
ax.set_title("Customer percentage of responses to offer (0=no, 1=yes)")
plt.savefig(output_dir / "response_100pc_bar.png", dpi=150, bbox_inches='tight')
show_plot(fig, ax)